In [2]:
import requests
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

df = pd.read_csv("Opened_OSHPD_Facilities.csv")
base_url = "https://earthquake.usgs.gov/ws/designmaps/asce7-22.json"
params_template = {
    "riskCategory": "IV",
    "siteClass": "Default",
    "title": "Example"
}
# param
results = []
failed_rows_722 = []
max_workers = 4       # Number of parallel threads
delay = 0.5           # Delay per request to prevent overloading API

In [4]:


def usgs_fetch(row, params_template, base_url, max_retries=3, delay=0.5):
    params = {
        **params_template,
        "latitude": row["Latitude"],
        "longitude": row["Longitude"]
    }

    for attempt in range(max_retries):
        try:
            response = requests.get(base_url, params=params, timeout=10)
            if response.status_code == 200:
                data = response.json()
                seismic = data.get("response", {}).get("data", {})
                sms = seismic.get("sms")
                sm1 = seismic.get("sm1")
              #  ss = seismic.get("ss")
              #  s1 = seismic.get("s1")
              #  sdc = seismic.get("sdc")
                
                #print - uncomment to get print status updates
                print(f"{row['Facility']} -> SMS: {sms}, SM1: {sm1}")

                

                return {
                    "Facility": row["Facility"],
                    "City": row["AddrCity"],
                    "Latitude": row["Latitude"],
                    "Longitude": row["Longitude"],
                    "SMS": sms,
                    "SM1": sm1,
                   # "SS": ss,
                   # "S1": s1,
                   # "SDC": sdc,
                }

            elif response.status_code == 429:
                # Rate limit hit
                print(f"Rate limit hit for {row['Facility']} (attempt {attempt + 1}) — sleeping 60 seconds")
                time.sleep(60)
                continue
            else:
                print(f"Attempt {attempt + 1} failed for {row['Facility']} with status {response.status_code}")

        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt + 1} exception for {row['Facility']}: {e}")

        time.sleep(delay)

    # If all retries fail, return placeholder and mark as failed
    failed_rows_722.append({
        "Facility": row["Facility"],
        "City": row["AddrCity"],
        "Latitude": row["Latitude"],
        "Longitude": row["Longitude"],
        "SMS": sms,
        "SM1": sm1,
        #"SS": ss,
        #"S1": s1,
        #"SDC": sdc,
            })

    return {

    "Facility": row["Facility"],
    "City": row["AddrCity"],
    "Latitude": row["Latitude"],
    "Longitude": row["Longitude"],
    "SMS": None,
    "SM1": None,
    #"SS": None,
    #"S1": None,
    #"SDC": None,
    }



In [5]:

# Run concurrent 
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(usgs_fetch, row, params_template, base_url) for _, row in df.iterrows()]
    for future in as_completed(futures):
        results.append(future.result())

#sds_results.sort(key=lambda x: x['Facility'])


10022 - Kaiser Foundation Hospital Sacramento -> SMS: 0.98, SM1: 0.63
10019 - Oroville Hospital -> SMS: 1.04, SM1: 0.65
10006 - Orchard Hospital -> SMS: 0.99, SM1: 0.64
10013 - Enloe Health -> SMS: 1.18, SM1: 0.73
10028 - Kaiser Foundation Hospital - San Marcos -> SMS: 1.34, SM1: 0.8510029 - Mark Twain Medical Center -> SMS: 0.72, SM1: 0.5

10027 - California Northstate University Medical Center -> SMS: 0.97, SM1: 0.63
10025 - UC Davis Rehabilitation Hospital -> SMS: 0.95, SM1: 0.62
10032 - Colusa Medical Center -> SMS: 1.15, SM1: 0.74
10035 - Stanford Health Care -> SMS: 2.27, SM1: 2.2
10036 - City of Hope Orange County Cancer Specialty Hospital -> SMS: 1.69, SM1: 1.11
10030 - River Vista Behavioral Health -> SMS: 0.95, SM1: 0.6
10037 - Southern Humboldt Community Healthcare District -> SMS: 2.62, SM1: 1.81
10039 - UCI Health - Irvine -> SMS: 1.7, SM1: 1.15
10040 - Harvest Medical Complex -> SMS: 2.33, SM1: 2.17
10041 - John George Psychiatric Hospital -> SMS: 2.52, SM1: 1.86
10044 - 

In [6]:
#results.sort(key=lambda x: x['Facility'])
# Save to CSV
pd.DataFrame(results).to_csv("Default_Risk_IV_Facilities_04_2026.csv", index=False)
pd.DataFrame(failed_rows_722).to_csv("failed_rows_Default_Risk_IV_Facilities_04_2026.csv", index=False)
